# 🎯 PWM, Motifs, and Sequence Logos

---

## Learning Objectives

- Understand Position Weight Matrices (PWM)
- Build and score sequence motifs
- Create and interpret sequence logos
- Search for motifs in sequences

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## What is a Motif?

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       SEQUENCE MOTIF                                    │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   A motif is a conserved pattern in biological sequences                │
│                                                                         │
│   Examples:                                                             │
│   • Transcription factor binding sites                                  │
│   • Splice sites (GT...AG)                                              │
│   • Protein functional domains                                          │
│   • Start codons (Kozak sequence)                                       │
│                                                                         │
│   Known binding sites for a TF:                                         │
│                                                                         │
│   Site 1:  A T G A C T C A                                              │
│   Site 2:  A T G A C T C A                                              │
│   Site 3:  A T G A C T T A                                              │
│   Site 4:  G T G A C T C A                                              │
│   Site 5:  A T G A C T C G                                              │
│            ─────────────────                                            │
│   Consensus: A T G A C T C A   (but position 1 and 8 vary)              │
│                                                                         │
│   Problem: Consensus loses information about variability                │
│   Solution: Position Weight Matrix (PWM)                                │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## Position Frequency Matrix (PFM)

```
┌─────────────────────────────────────────────────────────────────────────┐
│                   BUILDING A PFM                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Input sequences:                                                      │
│   1. ATGACTCA                                                           │
│   2. ATGACTCA                                                           │
│   3. ATGACTTA                                                           │
│   4. GTGACTCA                                                           │
│   5. ATGACTCG                                                           │
│                                                                         │
│   Position Frequency Matrix (counts):                                   │
│        Pos: 1   2   3   4   5   6   7   8                               │
│         A:  4   0   0   5   0   0   0   4                               │
│         C:  0   0   0   0   5   0   4   0                               │
│         G:  1   0   5   0   0   0   0   1                               │
│         T:  0   5   0   0   0   5   1   0                               │
│                                                                         │
│   Position Probability Matrix (frequencies):                            │
│        Pos: 1    2    3    4    5    6    7    8                        │
│         A: 0.8  0.0  0.0  1.0  0.0  0.0  0.0  0.8                       │
│         C: 0.0  0.0  0.0  0.0  1.0  0.0  0.8  0.0                       │
│         G: 0.2  0.0  1.0  0.0  0.0  0.0  0.0  0.2                       │
│         T: 0.0  1.0  0.0  0.0  0.0  1.0  0.2  0.0                       │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np

class PositionWeightMatrix:
    """
    Position Weight Matrix for DNA motifs.
    """
    BASES = ['A', 'C', 'G', 'T']
    
    def __init__(self, sequences=None):
        self.pfm = None  # Position Frequency Matrix (counts)
        self.ppm = None  # Position Probability Matrix
        self.pwm = None  # Position Weight Matrix (log-odds)
        self.length = 0
        
        if sequences:
            self.build(sequences)
    
    def build(self, sequences, pseudocount=0.1):
        """
        Build PWM from aligned sequences.
        
        Parameters:
            sequences: List of equal-length DNA strings
            pseudocount: Small value to avoid log(0)
        """
        if not sequences:
            return
        
        self.length = len(sequences[0])
        n_seq = len(sequences)
        
        # Build Position Frequency Matrix (counts)
        self.pfm = np.zeros((4, self.length))
        
        for seq in sequences:
            for pos, base in enumerate(seq.upper()):
                if base in self.BASES:
                    self.pfm[self.BASES.index(base), pos] += 1
        
        # Add pseudocounts and convert to probabilities
        self.ppm = (self.pfm + pseudocount) / (n_seq + 4 * pseudocount)
        
        # Convert to log-odds (PWM)
        background = 0.25  # Assume equal base frequencies
        self.pwm = np.log2(self.ppm / background)
    
    def score_sequence(self, sequence):
        """
        Score a sequence using the PWM.
        
        Returns log-odds score (higher = better match).
        """
        if len(sequence) != self.length:
            raise ValueError(f"Sequence length must be {self.length}")
        
        score = 0.0
        for pos, base in enumerate(sequence.upper()):
            if base in self.BASES:
                score += self.pwm[self.BASES.index(base), pos]
            else:
                score += 0  # Unknown base contributes nothing
        
        return score
    
    def max_score(self):
        """Return maximum possible score."""
        return np.sum(np.max(self.pwm, axis=0))
    
    def min_score(self):
        """Return minimum possible score."""
        return np.sum(np.min(self.pwm, axis=0))

# Example
sites = [
    "ATGACTCA",
    "ATGACTCA",
    "ATGACTTA",
    "GTGACTCA",
    "ATGACTCG"
]

pwm = PositionWeightMatrix(sites)

print("Position Frequency Matrix (counts):")
print(f"     {' '.join(f'P{i+1:d}' for i in range(pwm.length))}")
for i, base in enumerate(pwm.BASES):
    print(f"  {base}: {' '.join(f'{int(c):2d}' for c in pwm.pfm[i])}")

print(f"\nScore range: {pwm.min_score():.2f} to {pwm.max_score():.2f}")

In [ ]:
def scan_sequence(sequence, pwm, threshold=None):
    """
    Scan a sequence for PWM matches.
    
    Returns list of (position, score) tuples.
    """
    if threshold is None:
        threshold = 0.5 * pwm.max_score()  # Default: 50% of max
    
    hits = []
    seq_len = len(sequence)
    
    for pos in range(seq_len - pwm.length + 1):
        subseq = sequence[pos:pos + pwm.length]
        score = pwm.score_sequence(subseq)
        
        if score >= threshold:
            hits.append((pos, subseq, score))
    
    return sorted(hits, key=lambda x: -x[2])  # Sort by score descending

# Example: scan for TATA-box like motif
test_seq = "ACGTATATAAACGTACGATAGACTCAGTGTGACTCAACGT"
#                   ^^^^^^^^ potential hit          ^^^^^^^^ another

hits = scan_sequence(test_seq, pwm, threshold=pwm.max_score() * 0.7)

print(f"Scanning: {test_seq}")
print(f"Motif length: {pwm.length}")
print(f"\nHits (score >= 70% max):")
for pos, subseq, score in hits:
    print(f"  Position {pos:2d}: {subseq} (score: {score:.2f})")

---

## Sequence Logo

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    SEQUENCE LOGO                                        │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Visual representation of sequence conservation                        │
│                                                                         │
│   Height = Information Content (bits)                                   │
│                                                                         │
│   2.0 │      G              T                                           │
│       │      │              │                                           │
│   1.5 │      G    A    C    T    C                                      │
│       │      │    │    │    │    │                                      │
│   1.0 │  A   G    A    C    T    C    A                                 │
│       │  │   │    │    │    │    │    │                                 │
│   0.5 │  A   G    A    C    T    C    A                                 │
│       │  G        │              T                                      │
│   0.0 └─────────────────────────────────                                │
│         1   2    3    4    5    6    7    8                             │
│                                                                         │
│   Information Content at position i:                                    │
│   IC(i) = 2 - H(i)                                                      │
│   where H(i) = -Σ p(b) × log₂(p(b))                                     │
│                                                                         │
│   Max IC = 2 bits (100% conserved)                                      │
│   Min IC = 0 bits (equal frequencies)                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def calculate_information_content(pwm):
    """
    Calculate information content at each position.
    
    Returns array of IC values (0-2 bits for DNA).
    """
    ic = np.zeros(pwm.length)
    
    for pos in range(pwm.length):
        # Shannon entropy
        probs = pwm.ppm[:, pos]
        entropy = -np.sum(probs * np.log2(probs + 1e-10))
        
        # Information content = max entropy - observed entropy
        max_entropy = 2  # log2(4) for DNA
        ic[pos] = max_entropy - entropy
    
    return ic

def ascii_logo(pwm, height=6):
    """
    Create simple ASCII representation of sequence logo.
    """
    ic = calculate_information_content(pwm)
    
    # Build logo rows
    rows = [['.' for _ in range(pwm.length)] for _ in range(height)]
    
    for pos in range(pwm.length):
        # Get bases sorted by probability
        probs = pwm.ppm[:, pos]
        sorted_idx = np.argsort(probs)[::-1]
        
        # Fill column based on IC
        filled_height = int(ic[pos] / 2 * height)
        
        current_row = height - 1
        for idx in sorted_idx:
            base = pwm.BASES[idx]
            base_height = int(probs[idx] * filled_height)
            
            for _ in range(base_height):
                if current_row >= 0:
                    rows[current_row][pos] = base
                    current_row -= 1
    
    # Print logo
    print("Sequence Logo (ASCII):")
    print("IC │")
    for i, row in enumerate(rows):
        ic_val = (height - i) / height * 2
        print(f"{ic_val:.1f}│ {' '.join(row)}")
    print(f"   └{'─' * (pwm.length * 2)}")
    print(f"     {' '.join(str(i+1) for i in range(pwm.length))}")

# Show logo for our PWM
ic = calculate_information_content(pwm)
print("Information Content per position:")
for pos in range(pwm.length):
    bar = '█' * int(ic[pos] * 10)
    print(f"  Position {pos+1}: {ic[pos]:.2f} bits {bar}")

print(f"\nTotal Information: {sum(ic):.2f} bits")

---

## Common Motif Databases

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    MOTIF DATABASES                                      │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   Database    │ Content              │ Format                           │
│   ────────────┼──────────────────────┼────────────────────────          │
│   JASPAR      │ TF binding motifs    │ PFM, MEME                        │
│   TRANSFAC    │ TF binding sites     │ TRANSFAC matrix                  │
│   HOCOMOCO    │ Human/mouse TF PWMs  │ PCM, PWM                         │
│   CIS-BP      │ TF binding motifs    │ Multiple formats                 │
│   MEME Suite  │ Discovered motifs    │ MEME format                      │
│                                                                         │
│   JASPAR matrix format example:                                         │
│   >MA0004.1 Arnt                                                        │
│   A [  4 19  0  0  0  0 ]                                               │
│   C [  4  0  0  0  0  0 ]                                               │
│   G [  3  0 23  0 23  0 ]                                               │
│   T [ 12  4  0 23  0 23 ]                                               │
│                                                                         │
│   MEME format example:                                                  │
│   MOTIF MA0004.1 Arnt                                                   │
│   letter-probability matrix: alength= 4 w= 6                            │
│   0.174  0.174  0.130  0.522                                            │
│   0.826  0.000  0.000  0.174                                            │
│   0.000  0.000  1.000  0.000                                            │
│   ...                                                                   │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def parse_jaspar(jaspar_text):
    """
    Parse JASPAR format PWM.
    
    Returns PositionWeightMatrix object.
    """
    lines = jaspar_text.strip().split('\n')
    
    name = lines[0].lstrip('>') if lines[0].startswith('>') else 'Unknown'
    
    # Parse count lines
    import re
    counts = {}
    
    for line in lines[1:]:
        match = re.match(r'([ACGT])\s*\[([\d\s.]+)\]', line)
        if match:
            base = match.group(1)
            values = [float(x) for x in match.group(2).split()]
            counts[base] = values
    
    if not counts:
        return None
    
    # Convert to sequences for PWM builder
    length = len(counts['A'])
    pwm = PositionWeightMatrix()
    pwm.length = length
    pwm.pfm = np.array([counts[b] for b in pwm.BASES])
    
    # Normalize
    total = pwm.pfm.sum(axis=0)
    pwm.ppm = (pwm.pfm + 0.1) / (total + 0.4)
    pwm.pwm = np.log2(pwm.ppm / 0.25)
    
    return pwm, name

# Example JASPAR matrix (Arnt)
jaspar_example = """>MA0004.1 Arnt
A [  4 19  0  0  0  0 ]
C [  4  0  0  0  0  0 ]
G [  3  0 23  0 23  0 ]
T [ 12  4  0 23  0 23 ]"""

arnt_pwm, name = parse_jaspar(jaspar_example)
print(f"Parsed: {name}")
print(f"Length: {arnt_pwm.length}")
print(f"\nConsensus: {''.join(arnt_pwm.BASES[i] for i in np.argmax(arnt_pwm.ppm, axis=0))}")

---

## 🏋️ Exercises

### Exercise 1: Build PWM from FASTA
Read aligned sequences from FASTA and build a PWM.

### Exercise 2: Reverse Complement Search
Extend the scanner to search both strands.

### Exercise 3: De novo Motif Finding
Implement a simple expectation-maximization motif finder.

---

## 📚 Resources

- [JASPAR](https://jaspar.genereg.net/) - TF binding profiles
- [MEME Suite](https://meme-suite.org/) - Motif discovery tools
- [WebLogo](https://weblogo.berkeley.edu/) - Sequence logo generator

---
**Navigation:** [Back to Course README](../../README.md) · [Open this notebook path](`Course/Archive/07_Kodomo_Structural_Bioinformatics/12_Motifs_Domains/01_PWM_Sequence_Motifs.ipynb`)
